In [1]:
import re
import datetime as dt
import pandas as pd
import pyreadr
import numpy as np
from datetime import timedelta, datetime

import warnings
warnings.filterwarnings("ignore")

import gc
gc.collect()
gc.collect()
gc.collect()


0

In [2]:
df= pd.read_csv(r"D:\TASK\HL & LAP\LAP\LAP files\LAP MOM\LAP_All_disb_agreements_2006_Nov'2025_Final.csv")
df= df[['AGREEMENTNO','EFF_RATE','LTV_bin', "AMTFIN","AGR AUTH DATE","STATUS_DESC","BRANCHDESC",
        "Region","State","Ticket size_bin", "Zone", "Area", "BCG INDUSTRYDESC",
        "BCG SUB INDUSTRYDESC","PROMOTIONDESC"]]
print(df.shape)
df.head(3)

(194886, 15)


,AGREEMENTNO,EFF_RATE,LTV_bin,AMTFIN,AGR AUTH DATE,STATUS_DESC,BRANCHDESC,Region,State,Ticket size_bin,Zone,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC
0,X0HECHE00000016515,12.00,4. 50%-60%,2400000,26/10/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,2. 20L-50L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN
1,X0HECHE00000016528,14.85,4. 50%-60%,2400000,26/10/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,2. 20L-50L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN
2,X0HECHE00000021140,15.15,2. 20%-40%,600000,14/11/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,1. <=20L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN


In [3]:
# import pandas as pd
# df2= pd.read_excel(r"D:\TASK\LAP MOM Dashboard\Dashboard_data\LAP_MOM_Disbst.xlsx")
# df2= df2[["AGREEMENTNO","AMTFIN","AGR AUTH DATE","STATUS_DESC","BRANCHDESC", "Region", "State","Ticket size_bin", "Zone", "Area", "PROMOTIONDESC", "BCG INDUSTRYDESC","BCG SUB INDUSTRYDESC"]]
# print(df2.shape)
# df2.head(3)
# df= df2.merge(df1, how='inner', on='AGREEMENTNO')
# df.shape

In [4]:
# Filtering active and closed cases
df= df.loc[df['STATUS_DESC'].str.contains('ACTIVE|CLOSED')]
print(df.shape)
df.head(3)

(183654, 15)


,AGREEMENTNO,EFF_RATE,LTV_bin,AMTFIN,AGR AUTH DATE,STATUS_DESC,BRANCHDESC,Region,State,Ticket size_bin,Zone,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC
0,X0HECHE00000016515,12.00,4. 50%-60%,2400000,26/10/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,2. 20L-50L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN
1,X0HECHE00000016528,14.85,4. 50%-60%,2400000,26/10/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,2. 20L-50L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN
2,X0HECHE00000021140,15.15,2. 20%-40%,600000,14/11/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,1. <=20L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN


In [5]:
df['AGR AUTH DATE'] = pd.to_datetime(df['AGR AUTH DATE'])

# Filter rows where 'AGR_AUTH_DATE' is greater than 1st April 2022
df = df[df['AGR AUTH DATE'] >= pd.to_datetime('2023-04-01', format='%Y-%m-%d')]
df["AGR AUTH DATE"].min()

Timestamp('2023-04-11 00:00:00')

In [6]:
df["AGR AUTH DATE"].max()

Timestamp('2025-11-30 00:00:00')

In [7]:
df.shape

(80405, 15)

### Loading All_cols Deliq file

In [8]:
dl= pd.read_csv(r"D:\TASK\HE_HL_allcols_dump_Rcode\output HE & HL\HE_HL_deldump_cash_flow_mar21_nov25_AllCol.csv", encoding='latin1')
print(dl.shape)
dl.head(3)

(8003158, 135)


,EMI_STARTDATE,LAST_EMI_DATE,EXPIRYDT,INTCOMP_BILLED,INTCOMP_RECD,ACCRUEDAMT,SECURITSATION_BANK,HYPOTHECATION_BANK,AGREEMENTNO,PROPOSALID,...,ADMIN.AND.PROCESSING.FEE.DUE,ADMIN.AND.PROCESSING.FEE.DPD,LEGAL.OR.RECOVERY.CHARGES,SD_AMT,SWITCH_CHARGES,EOM_MONTH,src,bkt,DPD,OD_PLUS_POS
0,05/09/2025,05/08/2026,05/09/2026,6047,6047,"17,489",NaN,LNB/2023-24/1049,EF01ABA0000067662,"8,00,26,311",...,NaN,NaN,NaN,NaN,NaN,Aug-25,ENCORE,0.0,0,1212735.0
1,05/09/2025,05/08/2026,05/09/2026,0,0,"4,319",NaN,NaN,EF01ABA0000067662,"8,00,26,311",...,NaN,NaN,NaN,NaN,NaN,Jul-25,ENCORE,0.0,0,1212735.0
2,05-09-2025,05-08-2026,05-09-2026,42354,42354,51030,NaN,LNB/2023-24/1049,EF01ABA0000067662,80026311,...,NaN,NaN,NaN,NaN,NaN,NOV-25,ENCORE,0.0,0,924088.0


### EMI_STARTDATE Mapping

In [9]:
# Taking recent month emi date and mapping against agreementno's

dl['EMI_STARTDATE']= pd.to_datetime(dl['EMI_STARTDATE'],  errors='coerce')
dl['EMI_STARTDATE']= dl['EMI_STARTDATE'].dt.strftime('%Y-%m-%d')

emi1= dl.loc[(dl['month']=='nov-2025'), ['AGREEMENTNO','EMI_STARTDATE']]
rough1= emi1.merge(df['AGREEMENTNO'], on='AGREEMENTNO', how='right')
print(rough1.shape)
rough1.head(3)

(80405, 2)


,AGREEMENTNO,EMI_STARTDATE
0,HE01AAJ00000041667,NaN
1,HE01AAJ00000041684,NaN
2,HE01AAJ00000041964,NaN


In [10]:
# mapping the emi_start date for closed cases agreements

emi2 = dl.drop_duplicates(subset='AGREEMENTNO', keep='last')
emi2= emi2[['AGREEMENTNO','EMI_STARTDATE']]
emi2.shape

(395581, 2)

In [11]:
rough2= rough1.merge(emi2, on='AGREEMENTNO', how='left')
print(rough2.shape)
rough2.head(3)

(80405, 3)


,AGREEMENTNO,EMI_STARTDATE_x,EMI_STARTDATE_y
0,HE01AAJ00000041667,NaN,2023-05-06
1,HE01AAJ00000041684,NaN,2023-05-08
2,HE01AAJ00000041964,NaN,2023-05-06


In [12]:
rough2.isna().sum()

AGREEMENTNO            0
EMI_STARTDATE_x    80405
EMI_STARTDATE_y     6429
dtype: int64

In [13]:
rough2['EMI_STARTDATE_x'] = rough2.apply(lambda row: row['EMI_STARTDATE_y'] if pd.isnull(row['EMI_STARTDATE_x']) else row['EMI_STARTDATE_x'], axis=1)
rough2.isna().sum()

AGREEMENTNO           0
EMI_STARTDATE_x    6429
EMI_STARTDATE_y    6429
dtype: int64

In [14]:
emi= rough2[['AGREEMENTNO','EMI_STARTDATE_x']]
col={'EMI_STARTDATE_x':'EMI_STARTDATE'}
emi= emi.rename(columns=col)
emi.head(3)

,AGREEMENTNO,EMI_STARTDATE
0,HE01AAJ00000041667,2023-05-06
1,HE01AAJ00000041684,2023-05-08
2,HE01AAJ00000041964,2023-05-06


In [15]:
#"PROP_TYPE","PROP_DESC"

data= df.merge(emi, how='left', on='AGREEMENTNO')
data.shape

(80405, 16)

### Taking PRODUCT SCHEME, DISB_STATUS from Deliq all cols

In [16]:
# mapping the product, scheme column against agreements

dfs1 = dl.drop_duplicates(subset='AGREEMENTNO', keep='last')
data = data.merge(dfs1[['AGREEMENTNO','PRODUCT', 'SCHEME', 'CUSTOMERNAME']], on='AGREEMENTNO', how='left')

In [17]:
# mapping the disb_status column

dfs2 = dl.loc[(dl['month']=='nov-2025'), ['AGREEMENTNO','DISB_STATUS']]
data= data.merge(dfs2[['AGREEMENTNO','DISB_STATUS']], on='AGREEMENTNO', how='left')

In [18]:
data = data.merge(dfs1[['AGREEMENTNO', 'DISB_STATUS']],  on='AGREEMENTNO', how='left')
data['DISB_STATUS_x'] = data.apply(lambda row: row['DISB_STATUS_y'] if pd.isnull(row['DISB_STATUS_x']) else row['DISB_STATUS_x'], axis=1)
data= data.drop('DISB_STATUS_y', axis=1)
col={'DISB_STATUS_x':'DISBS_STATUS'}
data= data.rename(columns=col)
data.head(3)

,AGREEMENTNO,EFF_RATE,LTV_bin,AMTFIN,AGR AUTH DATE,STATUS_DESC,BRANCHDESC,Region,State,Ticket size_bin,Zone,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC,EMI_STARTDATE,PRODUCT,SCHEME,CUSTOMERNAME,DISBS_STATUS
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,"Agriculture, forestry, fishing and related act...",Registered Mortgage,2023-05-06,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED
1,HE01AAJ00000041684,13.0,3. 40%-50%,2800000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,Readymade Garments/Clothes,BT,2023-05-08,LAP,Loan Against Property,MD ALAM MD JAN,FULLY DISBURSED
2,HE01AAJ00000041964,12.0,3. 40%-50%,2000000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,General Merchandise Store,Fresh Loans,2023-05-06,LAP,Loan Against Property,SUSHIL KUMAR GAURI SAH,FULLY DISBURSED


In [19]:
data.isnull().sum()

AGREEMENTNO                 0
EFF_RATE                    0
LTV_bin                     0
AMTFIN                      0
AGR AUTH DATE               0
STATUS_DESC                 0
BRANCHDESC                  0
Region                      0
State                       0
Ticket size_bin             0
Zone                        0
Area                        0
BCG INDUSTRYDESC         9737
BCG SUB INDUSTRYDESC    14413
PROMOTIONDESC               1
EMI_STARTDATE            6429
PRODUCT                     0
SCHEME                      0
CUSTOMERNAME                0
DISBS_STATUS                0
dtype: int64

In [20]:
column={'EFF_RATE':'CUSTOMER_IRR', 'STATUS_DESC':'STATUS',  'AGR AUTH DATE':'AGR_AUTH_DATE'} 

data= data.rename(columns=column)
data.head(3)

,AGREEMENTNO,CUSTOMER_IRR,LTV_bin,AMTFIN,AGR_AUTH_DATE,STATUS,BRANCHDESC,Region,State,Ticket size_bin,Zone,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC,EMI_STARTDATE,PRODUCT,SCHEME,CUSTOMERNAME,DISBS_STATUS
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,"Agriculture, forestry, fishing and related act...",Registered Mortgage,2023-05-06,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED
1,HE01AAJ00000041684,13.0,3. 40%-50%,2800000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,Readymade Garments/Clothes,BT,2023-05-08,LAP,Loan Against Property,MD ALAM MD JAN,FULLY DISBURSED
2,HE01AAJ00000041964,12.0,3. 40%-50%,2000000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,General Merchandise Store,Fresh Loans,2023-05-06,LAP,Loan Against Property,SUSHIL KUMAR GAURI SAH,FULLY DISBURSED


In [21]:
data['DISBS_STATUS'].nunique()

2

### Work on the EMI_STARTDATE for partially disbursed cases & null 

In [22]:
data['AGR_AUTH_DATE']= pd.to_datetime(data['AGR_AUTH_DATE'], format="%Y-%m-%d")
data['EMI_STARTDATE']= pd.to_datetime(data['EMI_STARTDATE'], format="%Y-%d-%m").dt.strftime('%Y-%m-%d')

In [23]:
## Code to create a one month ahead from agr_auth_date

data['AGR_AUTH_DATE']= pd.to_datetime(data['AGR_AUTH_DATE'], format="%Y-%m-%d")

def calculate_one_month_ahead(date):
    next_month = date + timedelta(days=30)
    next_month_5th = datetime(next_month.year, next_month.month, 5)
    return next_month_5th

data['one_month_ahead_flag'] = data['AGR_AUTH_DATE'].apply(calculate_one_month_ahead)

#data['one_month_ahead_flag'] = data['AGR_AUTH_DATE']+ pd.DateOffset(months=1)

data.head(3)

,AGREEMENTNO,CUSTOMER_IRR,LTV_bin,AMTFIN,AGR_AUTH_DATE,STATUS,BRANCHDESC,Region,State,Ticket size_bin,...,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC,EMI_STARTDATE,PRODUCT,SCHEME,CUSTOMERNAME,DISBS_STATUS,one_month_ahead_flag
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,MUZAFFARPUR AREA,Retail / Wholesale Trade,"Agriculture, forestry, fishing and related act...",Registered Mortgage,2023-06-05,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED,2023-05-05
1,HE01AAJ00000041684,13.0,3. 40%-50%,2800000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,...,MUZAFFARPUR AREA,Retail / Wholesale Trade,Readymade Garments/Clothes,BT,2023-08-05,LAP,Loan Against Property,MD ALAM MD JAN,FULLY DISBURSED,2023-05-05
2,HE01AAJ00000041964,12.0,3. 40%-50%,2000000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,MUZAFFARPUR AREA,Retail / Wholesale Trade,General Merchandise Store,Fresh Loans,2023-06-05,LAP,Loan Against Property,SUSHIL KUMAR GAURI SAH,FULLY DISBURSED,2023-05-05


In [24]:
# For EMI StartDate null cases and DISB_STATUS 'partially Disbursed' mark one_month_ahead_flag as EMI STARTDATE 

data['EMI_STARTDATE']= pd.to_datetime(data['EMI_STARTDATE'], format="%Y-%m-%d")
data['one_month_ahead_flag']= pd.to_datetime(data['one_month_ahead_flag'], format="%Y-%m-%d")

data['EMI_STARTDATE']= data['EMI_STARTDATE'].fillna(data['one_month_ahead_flag'])


In [25]:
data.isna().sum()

AGREEMENTNO                 0
CUSTOMER_IRR                0
LTV_bin                     0
AMTFIN                      0
AGR_AUTH_DATE               0
STATUS                      0
BRANCHDESC                  0
Region                      0
State                       0
Ticket size_bin             0
Zone                        0
Area                        0
BCG INDUSTRYDESC         9737
BCG SUB INDUSTRYDESC    14413
PROMOTIONDESC               1
EMI_STARTDATE               0
PRODUCT                     0
SCHEME                      0
CUSTOMERNAME                0
DISBS_STATUS                0
one_month_ahead_flag        0
dtype: int64

In [26]:
data['DISBS_STATUS']= data['DISBS_STATUS'].fillna('FULLY DISBURSED')
data['PRODUCT']= data['PRODUCT'].fillna('LAP')
data['SCHEME']= data['SCHEME'].fillna('HOME Equity-Finone')

In [27]:
data= data.fillna('Unmapped')

In [28]:
data= data.drop(['one_month_ahead_flag'], axis=1)
data.shape

(80405, 20)

In [29]:
data['PRODUCT'].value_counts()

PRODUCT
LAP     53233
MLAP    27172
Name: count, dtype: int64

In [30]:
data.to_excel(r'D:\TASK\HL & LAP\LAP\LAP Staticpool\LAP New\final.xlsx', index=False)
print("saved successfully!")

saved successfully!


In [31]:
#Loading final

final= pd.read_excel(r'D:\TASK\HL & LAP\LAP\LAP Staticpool\LAP New\final.xlsx')

In [32]:
final['AGR_AUTH_DATE']= pd.to_datetime(data['AGR_AUTH_DATE'], format="%d/%m/%Y")
final['EMI_STARTDATE']= pd.to_datetime(data['EMI_STARTDATE'], format="%d/%m/%Y")
final.head(5)

,AGREEMENTNO,CUSTOMER_IRR,LTV_bin,AMTFIN,AGR_AUTH_DATE,STATUS,BRANCHDESC,Region,State,Ticket size_bin,Zone,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC,EMI_STARTDATE,PRODUCT,SCHEME,CUSTOMERNAME,DISBS_STATUS
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,"Agriculture, forestry, fishing and related act...",Registered Mortgage,2023-06-05,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED
1,HE01AAJ00000041684,13.0,3. 40%-50%,2800000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,Readymade Garments/Clothes,BT,2023-08-05,LAP,Loan Against Property,MD ALAM MD JAN,FULLY DISBURSED
2,HE01AAJ00000041964,12.0,3. 40%-50%,2000000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,General Merchandise Store,Fresh Loans,2023-06-05,LAP,Loan Against Property,SUSHIL KUMAR GAURI SAH,FULLY DISBURSED
3,HE01AAJ00000042198,13.5,3. 40%-50%,2300000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,EAST,MUZAFFARPUR AREA,Manufacturing,Others,BT,2023-08-05,LAP,Loan Against Property,ANANT SAH JHINGAN SAH,FULLY DISBURSED
4,HE01AAK00000042819,12.0,3. 40%-50%,3000000,2023-04-30,ACTIVE,BHADRAK HE,ODISHA,ODISHA,2. 20L-50L,EAST,CUTTACK AREA,Agriculture,Crop production,BT,2023-09-05,LAP,Loan Against Property,PRADEEP KUMAR GHADAI,FULLY DISBURSED


### Base month file workings

In [33]:
base_month= dl[['AGREEMENTNO','INTCOMP_BILLED','INTCOMP_RECD','UNADJUST','EMI_OS','POS','DPD','OD_PLUS_POS','DISBURSEDAMT','CUSTOMERNAME','month']]
print(base_month.shape)
base_month.head(3)

(8003158, 11)


,AGREEMENTNO,INTCOMP_BILLED,INTCOMP_RECD,UNADJUST,EMI_OS,POS,DPD,OD_PLUS_POS,DISBURSEDAMT,CUSTOMERNAME,month
0,EF01ABA0000067662,6047,6047,0,0.0,1212735.0,0,1212735.0,"12,12,735",DAWAT FAMILY RESTAURANT,aug-2025
1,EF01ABA0000067662,0,0,0,0.0,1212735.0,0,1212735.0,"12,12,735",DAWAT FAMILY RESTAURANT,jul-2025
2,EF01ABA0000067662,42354,42354,0,0.0,924088.0,0,924088.0,1212735,DAWAT FAMILY RESTAURANT,nov-2025


In [34]:
base_month.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8003158 entries, 0 to 8003157
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   AGREEMENTNO     object 
 1   INTCOMP_BILLED  object 
 2   INTCOMP_RECD    object 
 3   UNADJUST        object 
 4   EMI_OS          float64
 5   POS             float64
 6   DPD             int64  
 7   OD_PLUS_POS     float64
 8   DISBURSEDAMT    object 
 9   CUSTOMERNAME    object 
 10  month           object 
dtypes: float64(3), int64(1), object(7)
memory usage: 671.7+ MB


In [35]:
base_month['UNADJUST'] = pd.to_numeric(base_month['UNADJUST'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
base_month['INTCOMP_BILLED'] = pd.to_numeric(base_month['INTCOMP_BILLED'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
base_month['INTCOMP_RECD'] = pd.to_numeric(base_month['INTCOMP_RECD'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
base_month['DISBURSEDAMT'] = pd.to_numeric(base_month['DISBURSEDAMT'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

In [36]:
cols_to_convert = ['POS', 'EMI_OS', 'UNADJUST', 'INTCOMP_BILLED', 'INTCOMP_RECD']

# Convert columns to numeric, replacing non-numeric entries with NaN
base_month[cols_to_convert] = base_month[cols_to_convert].apply(pd.to_numeric, errors='coerce')

In [37]:
base_month['POS_N'] = base_month.POS + base_month.EMI_OS - base_month.UNADJUST - base_month.INTCOMP_BILLED + base_month.INTCOMP_RECD
print(base_month.shape)
base_month.head(3)

(8003158, 12)


,AGREEMENTNO,INTCOMP_BILLED,INTCOMP_RECD,UNADJUST,EMI_OS,POS,DPD,OD_PLUS_POS,DISBURSEDAMT,CUSTOMERNAME,month,POS_N
0,EF01ABA0000067662,6047.0,6047.0,0.0,0.0,1212735.0,0,1212735.0,1212735.0,DAWAT FAMILY RESTAURANT,aug-2025,1212735.0
1,EF01ABA0000067662,0.0,0.0,0.0,0.0,1212735.0,0,1212735.0,1212735.0,DAWAT FAMILY RESTAURANT,jul-2025,1212735.0
2,EF01ABA0000067662,42354.0,42354.0,0.0,0.0,924088.0,0,924088.0,1212735.0,DAWAT FAMILY RESTAURANT,nov-2025,924088.0


In [38]:
base_month= final.merge(base_month[["AGREEMENTNO", "DPD", "POS_N", "DISBURSEDAMT", "month"]], on="AGREEMENTNO", how="inner").drop_duplicates()
base_month.shape

(1129836, 24)

In [39]:
base_month['month_date']= pd.to_datetime(base_month['month'], format="%b-%Y")
base_month['month_date'] = base_month['month_date'].dt.strftime('%Y-%m-%d')
base_month.head(2)

,AGREEMENTNO,CUSTOMER_IRR,LTV_bin,AMTFIN,AGR_AUTH_DATE,STATUS,BRANCHDESC,Region,State,Ticket size_bin,...,EMI_STARTDATE,PRODUCT,SCHEME,CUSTOMERNAME,DISBS_STATUS,DPD,POS_N,DISBURSEDAMT,month,month_date
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,2023-06-05,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED,0,99808.0,100000.0,apr-2023,2023-04-01
1,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,2023-06-05,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED,0,95520.0,100000.0,apr-2024,2024-04-01


In [40]:
base_month['month_date'] = pd.to_datetime(base_month['month_date'], format='%Y-%m-%d')
base_month['EMI_STARTDATE'] = pd.to_datetime(base_month['EMI_STARTDATE'], format='%Y-%m-%d')   
base_month['mob'] = ((base_month['month_date'].dt.month - base_month['EMI_STARTDATE'].dt.month)+
                     (base_month['month_date'].dt.year - base_month['EMI_STARTDATE'].dt.year)*12)
base_month.head(3)

,AGREEMENTNO,CUSTOMER_IRR,LTV_bin,AMTFIN,AGR_AUTH_DATE,STATUS,BRANCHDESC,Region,State,Ticket size_bin,...,PRODUCT,SCHEME,CUSTOMERNAME,DISBS_STATUS,DPD,POS_N,DISBURSEDAMT,month,month_date,mob
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED,0,99808.0,100000.0,apr-2023,2023-04-01,-2
1,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED,0,95520.0,100000.0,apr-2024,2024-04-01,10
2,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED,0,90299.0,100000.0,apr-2025,2025-04-01,22


In [41]:
base_month.isna().sum()

AGREEMENTNO             0
CUSTOMER_IRR            0
LTV_bin                 0
AMTFIN                  0
AGR_AUTH_DATE           0
STATUS                  0
BRANCHDESC              0
Region                  0
State                   0
Ticket size_bin         0
Zone                    0
Area                    0
BCG INDUSTRYDESC        0
BCG SUB INDUSTRYDESC    0
PROMOTIONDESC           0
EMI_STARTDATE           0
PRODUCT                 0
SCHEME                  0
CUSTOMERNAME            0
DISBS_STATUS            0
DPD                     0
POS_N                   0
DISBURSEDAMT            0
month                   0
month_date              0
mob                     0
dtype: int64

In [42]:
base_month['mob_order'] = base_month['mob']
base_month['mob'] = base_month['mob_order'].astype(str) + "M"

In [43]:
def dpd_stage(dpd):
    if dpd > 90:
        return "stage 3"
    elif 30 < dpd <= 90:
        return "stage 2"
    else:
        return "stage 1"
    
base_month['Value'] = base_month['DPD'].apply(dpd_stage)

In [44]:
base_month = base_month.loc[base_month.mob_order >=0 ]
print(base_month.shape)

(940372, 28)


In [45]:
base_month['mob_order'].unique()

array([10, 22,  2, 14, 26,  6, 18,  8, 20,  7, 19,  1, 13, 25,  0, 12, 24,
        9, 21, 11, 23,  5, 17, 29,  4, 16, 28,  3, 15, 27])

In [46]:
max=base_month['mob_order'].max()
print(max)

29


In [47]:
# Saving base month file
base_month.drop_duplicates().to_csv(r"D:\TASK\HL & LAP\LAP\LAP Staticpool\LAP New\base_month.csv", index=False)


### Static Pool Working

In [48]:
# Read CSV files
final = pd.read_excel(r"D:\TASK\HL & LAP\LAP\LAP Staticpool\LAP New\final.xlsx").drop_duplicates()
final['AGR_AUTH_DATE']= pd.to_datetime(data['AGR_AUTH_DATE'], format="%d/%m/%Y")
final['EMI_STARTDATE']= pd.to_datetime(data['EMI_STARTDATE'], format="%d/%m/%Y")

base_month = pd.read_csv(r"D:\TASK\HL & LAP\LAP\LAP Staticpool\LAP New\base_month.csv", 
                         usecols = ["AGREEMENTNO","DPD","mob_order","month"])
base_month = base_month.fillna(0)


# Define DPD stages mapping function
def dpd_stage(dpd):
    if dpd > 90:
        return "stage 3"
    elif 30 < dpd <= 90:
        return "stage 2"
    else:
        return "stage 1"

max= int(base_month['mob_order'].max())

# Process base_month data and merge with final
for m in range(max+1):
    mob = base_month[base_month['mob_order'] == m].copy()
    mob['DPD'] = mob['DPD'].astype(int)
    mob[str(m) + 'M_curr'] = mob['DPD'].apply(dpd_stage)

    final = final.merge(mob[['AGREEMENTNO', str(m) + 'M_curr']], on='AGREEMENTNO', how='left')
    final = final.drop_duplicates()
    print(m)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29


In [49]:
final.AGR_AUTH_DATE.max()

Timestamp('2025-11-30 00:00:00')

In [50]:
final.AGR_AUTH_DATE.min()

Timestamp('2023-04-11 00:00:00')

### Melting final to build LAP SPA base file

In [51]:
df = final.drop_duplicates()
df.columns

Index(['AGREEMENTNO', 'CUSTOMER_IRR', 'LTV_bin', 'AMTFIN', 'AGR_AUTH_DATE',
       'STATUS', 'BRANCHDESC', 'Region', 'State', 'Ticket size_bin', 'Zone',
       'Area', 'BCG INDUSTRYDESC', 'BCG SUB INDUSTRYDESC', 'PROMOTIONDESC',
       'EMI_STARTDATE', 'PRODUCT', 'SCHEME', 'CUSTOMERNAME', 'DISBS_STATUS',
       '0M_curr', '1M_curr', '2M_curr', '3M_curr', '4M_curr', '5M_curr',
       '6M_curr', '7M_curr', '8M_curr', '9M_curr', '10M_curr', '11M_curr',
       '12M_curr', '13M_curr', '14M_curr', '15M_curr', '16M_curr', '17M_curr',
       '18M_curr', '19M_curr', '20M_curr', '21M_curr', '22M_curr', '23M_curr',
       '24M_curr', '25M_curr', '26M_curr', '27M_curr', '28M_curr', '29M_curr'],
      dtype='object')

In [52]:
import re

id_vars=['AGREEMENTNO', 'CUSTOMER_IRR', 'LTV_bin', 'AMTFIN', 'AGR_AUTH_DATE',
       'STATUS', 'BRANCHDESC', 'Region', 'State', 'Ticket size_bin', 'Zone',
       'Area', 'BCG INDUSTRYDESC', 'BCG SUB INDUSTRYDESC', 'PROMOTIONDESC',
       'EMI_STARTDATE', 'PRODUCT', 'SCHEME', 'CUSTOMERNAME', 'DISBS_STATUS']


pattern =  re.compile(r'^\d+M_curr$')
value_var = [c for c in df.columns if pattern.match(c)]

# Sort value_vars numerically by the leading number (sorting in orders OM to currentM)
def month_num(col):
    return int(col.split('M_')[0])

value_vars= sorted(value_var, key=month_num)
value_vars

['0M_curr',
 '1M_curr',
 '2M_curr',
 '3M_curr',
 '4M_curr',
 '5M_curr',
 '6M_curr',
 '7M_curr',
 '8M_curr',
 '9M_curr',
 '10M_curr',
 '11M_curr',
 '12M_curr',
 '13M_curr',
 '14M_curr',
 '15M_curr',
 '16M_curr',
 '17M_curr',
 '18M_curr',
 '19M_curr',
 '20M_curr',
 '21M_curr',
 '22M_curr',
 '23M_curr',
 '24M_curr',
 '25M_curr',
 '26M_curr',
 '27M_curr',
 '28M_curr',
 '29M_curr']

In [53]:
## Keep adding M_curr in the value_vars=[ ] of code as replicating the df columns name. Avoid entering M_Curr manually for current month

df = pd.melt(df, id_vars= id_vars, 
              value_vars= value_vars, var_name='Attribute', value_name='Value')

df['mob_order'] = pd.to_numeric(df['Attribute'].str.extract(r'([\d.]+)M_curr')[0])
df['mob'] = df['mob_order'].astype(str) + "M"
df["conc"] = df[["AGREEMENTNO", "mob"]].apply(" ".join, axis=1)

In [54]:
pd.unique(df.Value)

array(['stage 1', nan, 'stage 3', 'stage 2'], dtype=object)

In [55]:
df.columns

Index(['AGREEMENTNO', 'CUSTOMER_IRR', 'LTV_bin', 'AMTFIN', 'AGR_AUTH_DATE',
       'STATUS', 'BRANCHDESC', 'Region', 'State', 'Ticket size_bin', 'Zone',
       'Area', 'BCG INDUSTRYDESC', 'BCG SUB INDUSTRYDESC', 'PROMOTIONDESC',
       'EMI_STARTDATE', 'PRODUCT', 'SCHEME', 'CUSTOMERNAME', 'DISBS_STATUS',
       'Attribute', 'Value', 'mob_order', 'mob', 'conc'],
      dtype='object')

In [56]:
# Taking POS_N, DisbursedAMt, PRODUCT, SCHEME, DISB_STATUS cols from base month file

base= pd.read_csv(r"D:\TASK\HL & LAP\LAP\LAP Staticpool\LAP New\base_month.csv", usecols=['DPD','POS_N', 'AGREEMENTNO', 'DISBURSEDAMT', 'mob', 'month'])
base["conc"] = base[["AGREEMENTNO", "mob"]].apply(" ".join, axis=1)
base.columns

Index(['AGREEMENTNO', 'DPD', 'POS_N', 'DISBURSEDAMT', 'month', 'mob', 'conc'], dtype='object')

In [57]:
# Doing the left join of few cols from base file with melted file

new = df.merge(base[[ 'POS_N', 'DISBURSEDAMT', 'DPD', 'month', 'conc']], on="conc", how = "left")

In [59]:
new.head(3)

,AGREEMENTNO,CUSTOMER_IRR,LTV_bin,AMTFIN,AGR_AUTH_DATE,STATUS,BRANCHDESC,Region,State,Ticket size_bin,...,DISBS_STATUS,Attribute,Value,mob_order,mob,conc,POS_N,DISBURSEDAMT,DPD,month
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,FULLY DISBURSED,0M_curr,stage 1,0,0M,HE01AAJ00000041667 0M,99613.0,100000.0,0.0,jun-2023
1,HE01AAJ00000041684,13.0,3. 40%-50%,2800000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,...,FULLY DISBURSED,0M_curr,stage 1,0,0M,HE01AAJ00000041684 0M,2794905.0,2800000.0,0.0,aug-2023
2,HE01AAJ00000041964,12.0,3. 40%-50%,2000000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,FULLY DISBURSED,0M_curr,stage 1,0,0M,HE01AAJ00000041964 0M,1982419.0,2000000.0,0.0,jun-2023


In [60]:
# Function to remove the "1." and "2." prefixes in Ticket Size Bin column

def remove_prefix(entry):
    return re.sub(r'\b[1-9]\.', '', entry)

new['LTV_bin'] = new['LTV_bin'].apply(lambda x: remove_prefix(str(x)))

In [61]:
new['LTV_bin']= new['LTV_bin'].fillna('Unmapped')
new['LTV_bin'].value_counts()

LTV_bin
50%-60%    646650
40%-50%    625020
20%-40%    615120
60%-70%    420600
<=20%      104580
>70%          180
Name: count, dtype: int64

In [62]:
col= {"STATUS":"STATUS_DESC", "Region":"REGION", "Zone":"ZONE", "Area":"AREA", "Ticket size_bin":"LOAN BAND", 
      "SCHEME":"SCHEMEDESC", "LTV_bin":"LTV BAND"}

new = new.rename(columns=col)
new.head(3)

,AGREEMENTNO,CUSTOMER_IRR,LTV BAND,AMTFIN,AGR_AUTH_DATE,STATUS_DESC,BRANCHDESC,REGION,State,LOAN BAND,...,DISBS_STATUS,Attribute,Value,mob_order,mob,conc,POS_N,DISBURSEDAMT,DPD,month
0,HE01AAJ00000041667,14.0,40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,FULLY DISBURSED,0M_curr,stage 1,0,0M,HE01AAJ00000041667 0M,99613.0,100000.0,0.0,jun-2023
1,HE01AAJ00000041684,13.0,40%-50%,2800000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,...,FULLY DISBURSED,0M_curr,stage 1,0,0M,HE01AAJ00000041684 0M,2794905.0,2800000.0,0.0,aug-2023
2,HE01AAJ00000041964,12.0,40%-50%,2000000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,FULLY DISBURSED,0M_curr,stage 1,0,0M,HE01AAJ00000041964 0M,1982419.0,2000000.0,0.0,jun-2023


In [63]:
# marking Unmapped for null values in BCG INDUSTRYDESC

new['BCG INDUSTRYDESC']= new['BCG INDUSTRYDESC'].fillna('Unmapped')

new['BCG INDUSTRYDESC'].value_counts()

BCG INDUSTRYDESC
Manufacturing                                        821460
Service Industry                                     443880
Retail / Wholesale Trade                             427260
Unmapped                                             292110
Consultancy / Professional Services                  211200
Service                                              103860
Agriculture                                           41250
Retail trade                                          28320
OTHER SERVICE ACTIVITIES                               9300
Salaried / Profession Codes                            9180
Wholesale trade                                        9030
Other Service Activities                               6960
Accommodation and food service activities              2370
Construction                                           1560
Job Work                                                780
Agriculture, forestry and fishing                       660
CONSTRUCTION           

In [64]:
new['AGR_AUTH_DATE'].max()

Timestamp('2025-11-30 00:00:00')

In [68]:
#base_month= base_month.fillna(0)
new.drop_duplicates().to_csv("D:\\TASK\\HL & LAP\\LAP\\LAP Staticpool\\LAP New\\LAP_SPA.csv", index = False)
print("saved successfully!!")

saved successfully!!
